In [1]:
import os
import pandas as pd
import kagglehub

# Load the dataset
path = kagglehub.dataset_download("gowrishankarp/newspaper-text-summarization-cnn-dailymail")

# List the files in dataset directory
print("files in dataset: ", os.listdir(path))

# Define paths for all three splits
train_path = os.path.join(path, 'cnn_dailymail', 'train.csv')
val_path = os.path.join(path, 'cnn_dailymail', 'validation.csv')
test_path = os.path.join(path, 'cnn_dailymail', 'test.csv')

# Load dataframes
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

# Display first 10 rows of train data
train_df.head(10)

Using Colab cache for faster access to the 'newspaper-text-summarization-cnn-dailymail' dataset.
files in dataset:  ['cnn_dailymail']


,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,0002c17436637c4fe1837c935c04de47adb18e9a,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,0003ad6ef0c37534f80b55b4235108024b407f0b,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...
5,0004306354494f090ee2d7bc5ddbf80b63e80de6,He's been accused of making many a fashion fau...,Prime Minister and his family are enjoying an ...
6,0005d61497d21ff37a17751829bd7e3b6e4a7c5c,By . Daily Mail Reporter . PUBLISHED: . 01:15 ...,NBA star calls for black and Hispanic communit...
7,0006021f772fad0aa78a977ce4a31b3faa6e6fe5,By . Daily Mail Reporter . This is the moment ...,London Midland service had been pulling into T...
8,00083697263e215e5e7eda753070f08aa374dd45,There are a number of job descriptions waiting...,Tony Pulis believes Saido Berahino should look...
9,000940f2bb357ac04a236a232156d8b9b18d1667,"Canberra, Australia (CNN) -- At first glance, ...",Black box data from Flight 370 could be analyz...


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load BART model and tokenizer
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Select one document from your data
article_text = train_df.iloc[0]['article']

# Tokenize text with truncation at 1024 tokens
inputs = tokenizer(article_text, max_length=1024, truncation=True, return_tensors="pt")

print("inputs", inputs)

# Generate the concise summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    min_length=40,
    num_beams=4,
    early_stopping=True
)

print("summary_ids", summary_ids)

# Decode back to text
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Summary: {summary}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

inputs {'input_ids': tensor([[    0,  2765,   479,  1562,   977,   479, 29731,  7976, 14849,  1691,
            35,   479,   501,    35,  1225, 12936,     6,   564,   779,  1014,
           479,  1721,   479,   121, 44964,    35,   479,   379,    35,  3367,
         12936,     6,   564,   779,  1014,   479,    20, 23766,     9,     5,
          8044,  4019,  3643, 12074,    11,   369,  6223,    34,  4924,  2905,
          2213,     9,  2352,   453,    11,  8044,     6,  2374,   286,  2258,
             8,  7832,   990,  3355,     7,     5, 24426,    83,  6793,    11,
           628,   772,     8,   419,   779,     4,    20,   194,  1309,   641,
            34,  1167,    41,  7640,     9,  4895,    13,  1268,    54,  2922,
           292, 10550,     8,   362, 42418,     4,  8163,   610, 41303,   102,
            36,  7711,    43,     9,     5,  8044,  4019,  3643, 12074,    11,
           369,  6223,    34,  4924,  2905,  2213,     9,  2352,   453,    11,
          8044,     6,  2374,  

In [3]:
!pip install rouge-score

In [4]:
from rouge_score import rouge_scorer

# Initialize the scorer for ROUGE-1, ROUGE-2, and ROUGE-L
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Define reference (ground truth) and candidate (model output)
reference = train_df.iloc[0]['highlights']
candidate = summary  # The summary generated in the previous step

# Calculate scores
scores = scorer.score(reference, candidate)

# Display results
print("Evaluation Metrics:")
for key, value in scores.items():
    print(f"{key}: Precision: {value.precision:.4f}, Recall: {value.recall:.4f}, F1: {value.fmeasure:.4f}")

Evaluation Metrics:
rouge1: Precision: 0.3617, Recall: 0.5000, F1: 0.4198
rouge2: Precision: 0.2391, Recall: 0.3333, F1: 0.2785
rougeL: Precision: 0.2979, Recall: 0.4118, F1: 0.3457


In [5]:
!pip install pytextrank

In [6]:
import spacy
import pytextrank

# Load spaCy and add PyTextRank to the pipeline
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")

# Process the article
article_text = train_df.iloc[0]['article']
doc = nlp(article_text)

# Examine the top 3 sentences ranked by TextRank
print("TextRank Summary:")
for sent in doc._.textrank.summary(limit_phrases=15, limit_sentences=3):
    print(sent)

/usr/local/lib/python3.12/dist-packages
TextRank Summary:
Bishop John Folda (pictured) of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo, Grand Forks and Jamestown to the hepatitis A .
The bishop of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo, Grand Forks and Jamestown to the hepatitis A virus in late September and early October.
The diocese announced on Monday that Bishop John Folda is taking time off after being diagnosed with hepatitis A.


In [7]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Sample small amounts for the demonstration
train_sub = train_df.sample(100)
val_sub = val_df.sample(20)

def tokenize_batch(row):
    # Modern text_target approach
    model_inputs = tokenizer(row['article'], max_length=1024, truncation=True, padding="max_length")
    labels = tokenizer(text_target=row['highlights'], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Setup training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    predict_with_generate=True,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=1,
    logging_steps=10,
    # Changed to 'eval_strategy' to match the latest Transformers update
    eval_strategy="steps"
)

# Prepare the datasets
train_dataset = train_sub.apply(tokenize_batch, axis=1).tolist()
eval_dataset = val_sub.apply(tokenize_batch, axis=1).tolist()

# Initialize and Run Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset, # uses validation set
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
10,4.179985,1.987658
20,1.588642,1.289452
30,1.138017,1.141687
40,1.006088,1.080400
50,0.984495,1.062488


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50, training_loss=1.7794455337524413, metrics={'train_runtime': 3031.023, 'train_samples_per_second': 0.033, 'train_steps_per_second': 0.016, 'total_flos': 216710460211200.0, 'train_loss': 1.7794455337524413, 'epoch': 1.0})

In [8]:
# Create the test dataset from test.csv
test_sub = test_df.sample(20)
test_dataset = test_sub.apply(tokenize_batch, axis=1).tolist()

# Use the trainer to evaluate on the final test set
results = trainer.evaluate(eval_dataset=test_dataset)

print(f"Final ROUGE scores: {results}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Final ROUGE scores: {'eval_loss': 1.1178066730499268, 'eval_runtime': 125.4789, 'eval_samples_per_second': 0.159, 'eval_steps_per_second': 0.024, 'epoch': 1.0}
